In [30]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import json


### Load initial curve

In [61]:
name = "ideal_frc/52_frc.txt"
knot = np.genfromtxt(name)
#knot[:,2] += 0.1*np.random.randn(len(knot))
len(knot)

81

### Reach equilibrium 

In [62]:
def save_closed(filename, traj):

    N = len(traj)
    
    FILE = open(filename,"w")
    FILE = open(filename,"a")

    min_x = 3*min(traj.T[0])/3
    max_x = 3*max(traj.T[0])/3

    min_y = 3*min(traj.T[1])/3
    max_y = 3*max(traj.T[1])/3

    min_z = 3*min(traj.T[2])/3
    max_z = 3*max(traj.T[2])/3    
    
    FILE.writelines("LAMMPS data file for polymer\n\n")
    FILE.writelines(str(N)+" atoms\n"+str(N)+" bonds\n"+str(N)+" angles\n\n")
    FILE.writelines(" 1 atom types\n 1 bond types\n 1 angle types\n\n\n")

    
    
    FILE.writelines(str(min_x)+ " " + str(max_x) + " xlo xhi"+ "\n")
    FILE.writelines(str(min_y)+ " " + str(max_y) +  " ylo yhi"+"\n")
    FILE.writelines(str(min_z)+ " " + str(max_z) +" zlo zhi"+ "\n\n\n")

    FILE.writelines("Masses\n\n")

    FILE.writelines(" 1 1\n")
    
    FILE.writelines("Atoms\n\n")

    for i in range(N):
        
        FILE.writelines("{} 1 1 {} {} {} 0 0 0\n".format(i+1, traj[i][0],traj[i][1],traj[i][2]))

    
    
    FILE.writelines("\nBonds\n\n")

    for i in range(0,N-1):
        FILE.writelines("{} 1 {} {}\n".format(i+1,i+1, i+2))
    FILE.writelines("{} 1 {} 1\n".format(N,N))
    FILE.write("\nAngles\n\n")
    for i in range(0,N-2):
        FILE.writelines("{} 1 {} {} {}\n".format(i+1,i+1, i+2,i+3))
    FILE.writelines("{} 1 {} {} 1\n".format(N-1,N-1,N))
    FILE.writelines("{} 1 {} 1 2\n".format(N,N))
    FILE.close()


In [63]:
save_closed("temporary/initial.txt", np.array(knot))

In [64]:
def initial(seed):
    !lmp_serial -var number {seed} -in equilibrium.txt

In [65]:
initial(3)

LAMMPS (2 Aug 2023 - Update 3)
Reading data file ...
  orthogonal box = (-7.3332157 -7.4301742 -2.8215376) to (7.2765366 7.6182315 2.2904504)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  81 atoms
  scanning bonds ...
  1 = max bonds/atom
  scanning angles ...
  1 = max angles/atom
  reading bonds ...
  81 bonds
  reading angles ...
  81 angles
Finding 1-2 1-3 1-4 neighbors ...
  special bond factors lj:    0        0        0       
  special bond factors coul:  0        0        0       
     2 = max # of 1-2 neighbors
     2 = max # of 1-3 neighbors
     4 = max # of 1-4 neighbors
     6 = max # of special neighbors
  special bonds CPU = 0.000 seconds
  read_data CPU = 0.001 seconds
Generated 0 of 0 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 1 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 2.12246
  ghost atom cutoff = 2.12246
  binsize = 1.06123, bins = 14 15 

In [66]:
def split_list(lst, size):
    return list(zip(*[iter(lst)] * size))    


def load_initial(seed):
    name = "temporary/initial_seed{}.xyz".format(seed)
    file = open(name)
    lines = file.readlines()
    curves = []
    N = int(lines[0])
    for line in lines:
        if line[0] == 'O':
            l = [float(a) for a in line.replace("\n", "").split(" ")[1:]]
            curves.append(l)

    X = np.array(split_list(curves, N))
    print(len(X))
    conf = X[-1] 
    return conf

In [67]:
traj = load_initial(3)
traj

1


array([[-1.31106e-01, -6.16353e+00, -4.02197e+00],
       [ 9.46766e-01, -6.21740e+00, -3.58765e+00],
       [ 1.90177e+00, -6.66057e+00, -3.54726e+00],
       [ 2.93989e+00, -6.28164e+00, -3.66067e+00],
       [ 4.02661e+00, -5.94340e+00, -3.23451e+00],
       [ 4.81728e+00, -5.33119e+00, -2.72771e+00],
       [ 5.18626e+00, -4.38932e+00, -2.24184e+00],
       [ 5.28718e+00, -3.46408e+00, -2.00553e+00],
       [ 5.65629e+00, -2.47164e+00, -1.83811e+00],
       [ 6.12019e+00, -1.71030e+00, -1.34458e+00],
       [ 6.65884e+00, -9.02257e-01, -7.10863e-01],
       [ 6.78437e+00, -1.64647e-01,  4.70614e-02],
       [ 6.79429e+00,  7.95253e-01,  6.21169e-01],
       [ 6.56326e+00,  1.98878e+00,  7.52743e-01],
       [ 5.92317e+00,  2.71980e+00,  1.18267e+00],
       [ 4.91833e+00,  2.95226e+00,  9.55783e-01],
       [ 3.91519e+00,  3.25520e+00,  8.15024e-01],
       [ 3.14395e+00,  3.47336e+00,  2.16886e-01],
       [ 2.45864e+00,  4.05375e+00, -2.75904e-01],
       [ 1.76035e+00,  4.96324e

In [68]:
fig = plt.figure(figsize = (7, 7))
ax = fig.add_subplot(projection='3d')
scat = ax.scatter(knot[:,1],knot[:,2], knot[:,0], s=10,edgecolors='black',alpha = 1)
ax.plot( knot[:,1],knot[:,2],knot[:,0], linewidth=1, c = 'black', label='Original curve')
scat = ax.scatter( traj[:,1],traj[:,2],traj[:,0],s=10,edgecolors='blue',alpha = 1, label='Curve at equilibruium')
ax.plot( traj[:,1],traj[:,2],traj[:,0], linewidth=3, c  ='blue')
plt.legend()
plt.show()

### Perturb the curve at equilibrium

In [69]:
save_closed("temporary/start.txt", np.array(traj))

In [70]:
def load_perturbation(seed, folder = 'temporary'):
    name = "{}/perturbation_seed{}.xyz".format(folder,seed)
    file = open(name)
    lines = file.readlines()
    curves = []
    N = int(lines[0])
    for line in lines:
        if line[0] == 'O':
            l = [float(a) for a in line.replace("\n", "").split(" ")[1:]]
            curves.append(l)

    X = np.array(split_list(curves, N))
    conf = X[-1] 
    return conf

In [71]:
def run(seed):
    !lmp_serial -var number {seed} -in perturbations.txt

In [72]:
for i in range(1,51):
    run(i)
    p = load_perturbation(i)


LAMMPS (2 Aug 2023 - Update 3)
Reading data file ...
  orthogonal box = (-7.25171 -6.66057 -4.02197) to (6.79429 8.26525 1.95247)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  81 atoms
  scanning bonds ...
  1 = max bonds/atom
  scanning angles ...
  1 = max angles/atom
  reading bonds ...
  81 bonds
  reading angles ...
  81 angles
Finding 1-2 1-3 1-4 neighbors ...
  special bond factors lj:    0        0        0       
  special bond factors coul:  0        0        0       
     2 = max # of 1-2 neighbors
     2 = max # of 1-3 neighbors
     4 = max # of 1-4 neighbors
     6 = max # of special neighbors
  special bonds CPU = 0.000 seconds
  read_data CPU = 0.001 seconds
Generated 0 of 0 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 1 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 2.12246
  ghost atom cutoff = 2.12246
  binsize = 1.06123, bins = 14 15 6
  1 neighb

In [73]:
from topoly import jones,alexander
for i in range(1,51):
    p = load_perturbation(i)
    poly = alexander([list(a) for a in p],closure=0)
    print(i,poly)


1 5_2
2 5_2
3 5_2
4 5_2
5 5_2
6 5_2
7 5_2
8 5_2
9 5_2
10 5_2
11 5_2
12 5_2
13 5_2
14 5_2
15 5_2
16 5_2
17 5_2
18 5_2
19 5_2
20 5_2
21 5_2
22 5_2
23 5_2
24 5_2
25 5_2
26 5_2
27 5_2
28 5_2
29 5_2
30 5_2
31 5_2
32 5_2
33 5_2
34 5_2
35 5_2
36 5_2
37 5_2
38 5_2
39 5_2
40 5_2
41 5_2
42 5_2
43 5_2
44 5_2
45 5_2
46 5_2
47 5_2
48 5_2
49 5_2
50 5_2


In [74]:
import plotly.graph_objects as go

fig = go.Figure()

for i in range(1, 5):
    p = load_perturbation(i)

    fig.add_trace(
        go.Scatter3d(
            x=p[:,1],
            y=p[:,2],
            z=p[:,0],
            mode="lines+markers",
            line=dict(width=6),
            marker=dict(size=3),
            name=f"perturbation {i}"
        )
    )

fig.update_layout(
    width=700,
    height=700,
    scene=dict(
        xaxis_title="y",
        yaxis_title="z",
        zaxis_title="x",
        aspectmode="data"
    )
)

fig.write_html('prove.html')

In [68]:
def average_crossing_number(p):
    """
    Polygonal approximation of the Gauss double integral.
    O(N^2), suitable for chains of a few hundred beads.
    """

    segs = p[1:] - p[:-1]
    mids = 0.5 * (p[1:] + p[:-1])

    acn = 0.0

    nseg = len(segs)

    for i in range(nseg):
        for j in range(i + 1, nseg):

            # skip adjacent segments
            if abs(i - j) <= 1:
                continue

            rij = mids[i] - mids[j]
            r = np.linalg.norm(rij)

            if r < 1e-12:
                continue

            acn += abs(
                np.dot(
                    np.cross(segs[i], segs[j]),
                    rij
                )
            ) / r**3

    return acn / (4 * np.pi)
def radius_of_gyration(p):
    com = p.mean(axis=0)
    return np.sqrt(np.mean(np.sum((p - com)**2, axis=1)))

for i in range(1, 50):
    p = load_perturbation(i)
    rog = radius_of_gyration(p)
    print(i, rog, average_crossing_number(p))

1 3.1636494545605633 7.240928443593992
2 3.465787366442496 6.553912100596519
3 3.664266790531014 6.2922517459875555
4 3.3316266625016935 6.722528882240595
5 3.536015219410115 6.443088271961996
6 3.106406291176693 7.759650208373536
7 2.998438528233062 8.591183292071895
8 3.6090120788188393 5.7991260092781935
9 3.3115369632704974 5.920850260991622
10 3.3192678850970947 6.629692969395922
11 3.8703983374114235 5.793689813209503
12 3.379213143659415 6.063351094695546
13 3.331418450054262 6.963175854687536
14 3.5356839112152505 6.212664152156382
15 3.3947119956572185 6.69129090361464
16 3.6227179727699914 6.373980048020896
17 3.7592688782679926 5.669908135770179
18 3.7124471760114046 5.480272172976618
19 3.633423654555864 5.971808520607473
20 3.572018494309227 7.088856303128371
21 4.447798316650997 5.307853498493195
22 3.7996509533850804 6.640199095584037
23 3.2919143312881856 7.11863785153613
24 3.7332562318364015 5.639997098446221
25 3.616704022113653 5.95503945760243
26 3.909610697785516 